# Sprint 2 - Limpeza, Tratamento, Split Temporal e EDA
Este notebook executa os Tópicos 1 e 2 detalhados no `Sprint2_TrilhaC.md`:
1. **Limpeza e Inspeção:** Verifica valores nulos, duplicatas e aplica regras de domínio (mantendo a pasta raw intocada).
2. **Split Temporal:** Divide os dados de 2015 a 2022 para Treino e 2023 a 2024 para Teste (Safra 2025 foi descartada pois a API do IBGE ainda não consolidou os dados).
3. **Análise Exploratória (EDA):** Realiza os gráficos requisitados APENAS sobre a base de Treino.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Garantir que a pasta interim existe
os.makedirs('../data/interim', exist_ok=True)

# Carregando os dados brutos da Sprint 1
df_raw = pd.read_csv('../data/raw/integrado.csv')

# Descartando a safra de 2025 conforme RFC (Dados do IBGE não consolidados)
df_raw = df_raw[df_raw['ano'] <= 2024]
display(df_raw.head())

## 1.1 Inspeção e Limpeza

In [ ]:
print("=== Estatística Descritiva (Raw) ===")
display(df_raw.describe())

print("\n=== Contagem de Nulos ===")
print(df_raw.isnull().sum())

print("\n=== Linhas Duplicadas ===")
print(df_raw.duplicated().sum())

**Diagnóstico do Grupo:** Conforme observado acima, nossa base integrada veio limpa da Etapa de Coleta (sem nulos e sem duplicatas), pois o tratamento primário de sentinelas (-999) já foi mapeado no momento da integração do IBGE e NASA na Sprint 1. E a safra de 2025 foi removida conforme acordado.

In [ ]:
df_limpo = df_raw.copy()

# Salvar em data/interim
df_limpo.to_csv('../data/interim/limpo.csv', index=False)
print("Dados limpos salvos em: ../data/interim/limpo.csv")

## 2. Divisão entre Treino e Teste (Split Temporal)
Para evitar *Data Leakage* temporal, vamos separar:
- **Treino:** 2015 a 2022
- **Teste:** 2023 a 2024

In [ ]:
df_treino = df_limpo[df_limpo['ano'] <= 2022].copy()
df_teste = df_limpo[df_limpo['ano'] >= 2023].copy()

print(f"Linhas no Treino: {df_treino.shape[0]} (Anos: {df_treino['ano'].min()} a {df_treino['ano'].max()})")
print(f"Linhas no Teste: {df_teste.shape[0]} (Anos: {df_teste['ano'].min()} a {df_teste['ano'].max()})")

## 4. Análise Exploratória de Dados (EDA) - Apenas no Treino!

In [ ]:
sns.set_theme(style="whitegrid")

# 1. Evolução da Produtividade (Treino)
plt.figure(figsize=(10, 4))
sns.lineplot(data=df_treino, x='ano', y='rendimento_kg_ha', marker='o', color='green')
plt.title('Evolução do Rendimento (Base de Treino)')
plt.xticks(df_treino['ano'])
plt.show()

# 2. Boxplot para encontrar Outliers
plt.figure(figsize=(6, 4))
sns.boxplot(y=df_treino['rendimento_kg_ha'], color='lightgreen')
plt.title('Boxplot de Rendimento (Treino)')
plt.show()

# 3. Matriz de Correlação das Features Climáticas x Rendimento
colunas_corr = ['rendimento_kg_ha', 'PRECTOTCORR', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'GWETROOT']
plt.figure(figsize=(8, 6))
sns.heatmap(df_treino[colunas_corr].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriz de Correlação (Base de Treino)')
plt.show()